download from here: https://tv.supa.sh/logs?c=dorozea

0. only need data when stream was on MANUAL
1. remove all the deleted messages MANUAL
    if deleted only new line and his message  up
2. remove the time outs and the banned persons MANUAL
3. remove #dorozea in the line
4. fix the date 
5. fix the capitalization 


fixed: 04.24.25 only first 10 line

In [1]:
import re
import pandas as pd
from collections import Counter
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt


In [2]:


# Load the file and clean the data
with open('testclean.txt', 'r', encoding='utf-8') as file:
    lines = file.readlines()

# Process each line
cleaned_lines = []
for line in lines:
    # Remove '#dorozea' and add 'UTC' to the timestamp
    if line.strip():  # Skip empty lines
        cleaned_line = line.replace('#dorozea ', '', 1).replace(']', ' UTC]', 1)
        cleaned_lines.append(cleaned_line)

# Save the cleaned data back to a file or print it
with open('cleaned_testclean.txt', 'w', encoding='utf-8') as file:
    file.writelines(cleaned_lines)

# Parse each cleaned line into components
parsed_data = []
for line in cleaned_lines:
    match = re.match(r'\[(.*?) UTC\] (\w+): (.*)', line)
    if match:
        date, user, message = match.groups()
        parsed_data.append([date, user, message])


In [3]:
# Read the list of filenames from the configuration file
with open('../../file_list.txt', 'r', encoding='utf-8') as config_file:
    file_names = config_file.read().splitlines()

# Regex pattern to match the data format
pattern = r'\[(.*?)\] (.*?): (.*)'

# Initialize an empty list to store parsed data
datalist = []
stream_count = 0
# Iterate over each specified file
for file in file_names:
    full_path = f"../../data/{file}"
    with open(full_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        for line in lines:
            match = re.match(pattern, line)
            if match:
                date, user, message = match.groups()
                datalist.append([date, user, message,stream_count])
    stream_count = stream_count + 1

# Create a DataFrame from the parsed data
data = pd.DataFrame(datalist, columns=["date", "user", "message","stream"])



data["user"] = data["user"].replace("Banties1g", "banties_x")
data["user"] = data["user"].replace("banties1g", "banties_x")
data["user"] = data["user"].replace("chili_poe", "chili_con_bacon")
data["user"] = data["user"].replace("CHILI_POE", "chili_con_bacon")
data["user"] = data["user"].replace("Chili_poe", "chili_con_bacon")
data["user"] = data["user"].replace("chili_conbacon", "chili_con_bacon")
data["user"] = data["user"].replace("Wirelesss_", "W1r3lesss")
data["user"] = data["user"].replace("treklul", "trek44_")
data["user"] = data["user"].replace("ttrek_", "trek44_")
data["user"] = data["user"].replace("trek_x", "trek44_")
data["user"] = data["user"].replace("TriplesingleJ", "TripleSingleJames")
data["user"] = data["user"].replace("uwu_cougar", "uuccugr")
data["user"] = data["user"].replace("uuccugr_","uuccugr")
data["user"] = data["user"].replace("StanIV4_", "stan_iv4")
data["user"] = data["user"].replace("Muuskie2", "Muuskie")
data["user"] = data["user"].replace("nishad_more1311", "nishad13")
data["user"] = data["user"].replace("softarballt", "softarr")
data["user"] = data["user"].replace("softarballtt23", "softarr")
data["user"] = data["user"].replace("lajosbarnabas", "lajoss__")
data["user"] = data["user"].replace("Bonkwiththefunk", "bonk67")

In [4]:
from collections import defaultdict

# Get all unique usernames
unique_users = data['user'].unique()

# Create a mapping from lowercase username to all variants

user_variants = defaultdict(set)
for user in unique_users:
    user_variants[user.lower()].add(user)

# Find usernames with different capitalization
duplicate_users = {k: v for k, v in user_variants.items() if len(v) > 1}
""" 
# Display the results

for lower, variants in duplicate_users.items():
    print(f"{lower}: {sorted(variants)}")
"""

' \n# Display the results\n\nfor lower, variants in duplicate_users.items():\n    print(f"{lower}: {sorted(variants)}")\n'

In [5]:
# Create a mapping from all variants to the canonical (sorted first) variant
variant_map = {}
for variants in duplicate_users.values():
    sorted_variants = sorted(variants)
    canonical = sorted_variants[0]
    for v in variants:
        variant_map[v] = canonical

# Replace usernames in 'user' column
data['user'] = data['user'].apply(lambda u: variant_map.get(u, u))


In [6]:
# Create the DataFrame
cleaned_lines_dataFrame = pd.DataFrame(parsed_data, columns=["date", "user", "message"])

cleaned_lines_dataFrame['date'] = pd.to_datetime(cleaned_lines_dataFrame['date'])

# not needed because it is done in the other files
#cleaned_lines_dataFrame['date'] = cleaned_lines_dataFrame['date'] + pd.Timedelta(hours=2)

In [7]:
# Create a mapping of lowercase -> properly capitalized usernames
user_mapping = {user.lower(): user for user in data['user'].unique()}

# Apply the mapping to the 'user' column in cleaned_lines_dataFrame
cleaned_lines_dataFrame['user'] = cleaned_lines_dataFrame['user'].apply(
    lambda u: user_mapping.get(u, u)
)

In [8]:
cleaned_lines_dataFrame.head(50)

,date,user,message
0,2026-04-09 13:59:39,polimpompis,EEK
1,2026-04-09 13:59:39,balintboss,Bello
2,2026-04-09 13:59:40,FilipStayout,yo
3,2026-04-09 13:59:40,nishad13,hi doro
4,2026-04-09 13:59:46,StreamElements,dorozea is now live! Streaming Just Chatting: ...
5,2026-04-09 13:59:51,FilipStayout,yello
6,2026-04-09 13:59:52,FlexGunships,woah wtf doro is hear aint no way omgbruh
7,2026-04-09 13:59:53,polimpompis,yellow intrigued
8,2026-04-09 13:59:53,Aluminiumminimumimmunity,hi
9,2026-04-09 13:59:55,flexus1771_,"agahi chat, hi doro"


In [9]:
# Copy user column before step 2
before_users = cleaned_lines_dataFrame['user'].copy()

In [10]:
import re

# Get the set of known lowercase usernames
known_users_lower = set(user_mapping.keys())

# Get unique unknown usernames from cleaned_lines_dataFrame
unknown_users = cleaned_lines_dataFrame[~cleaned_lines_dataFrame['user'].isin(user_mapping.values())]['user'].unique()

# Function to find proper capitalization of a user from messages
def find_capitalization_from_messages(username, messages):
    pattern = re.compile(r'\b' + re.escape(username) + r'\b', re.IGNORECASE)
    for msg in messages:
        match = pattern.search(msg)
        if match:
            # Return the actual matched string from message
            return match.group(0)
    return None

# Loop through unknown users and try to correct their capitalization
for user in unknown_users:
    messages = cleaned_lines_dataFrame[cleaned_lines_dataFrame['user'] == user]['message']
    corrected = find_capitalization_from_messages(user, messages)
    if corrected and corrected.lower() == user:
        cleaned_lines_dataFrame.loc[cleaned_lines_dataFrame['user'] == user, 'user'] = corrected


In [11]:
# Compare after step 2
changed = (before_users != cleaned_lines_dataFrame['user'])

# Count how many changed
num_changed = changed.sum()
print(f"Number of usernames changed in Step 2: {num_changed}")

# (Optional) Show which usernames changed
print(cleaned_lines_dataFrame.loc[changed, ['user']])

Number of usernames changed in Step 2: 1
           user
5730  arseNiic_


In [12]:
cleaned_lines_dataFrame.head(10)

,date,user,message
0,2026-04-09 13:59:39,polimpompis,EEK
1,2026-04-09 13:59:39,balintboss,Bello
2,2026-04-09 13:59:40,FilipStayout,yo
3,2026-04-09 13:59:40,nishad13,hi doro
4,2026-04-09 13:59:46,StreamElements,dorozea is now live! Streaming Just Chatting: ...
5,2026-04-09 13:59:51,FilipStayout,yello
6,2026-04-09 13:59:52,FlexGunships,woah wtf doro is hear aint no way omgbruh
7,2026-04-09 13:59:53,polimpompis,yellow intrigued
8,2026-04-09 13:59:53,Aluminiumminimumimmunity,hi
9,2026-04-09 13:59:55,flexus1771_,"agahi chat, hi doro"


In [13]:
# Extract the date from the first row and format it for the filename
filename_date = cleaned_lines_dataFrame['date'].iloc[0].strftime('%m-%d-%y')
filename = f'[{filename_date}].txt'

# Open the file in write mode
with open(filename, 'w', encoding='utf-8') as f:
    # Iterate over each row in the DataFrame
    for index, row in cleaned_lines_dataFrame.iterrows():
        # Format the date, user, and message into the desired string format
        # Adding 'UTC' as per the requested format, assuming the times are in UTC
        formatted_line = f"[{row['date'].strftime('%Y-%m-%d %H:%M:%S')} UTC] {row['user']}: {row['message']}\n"
        # Write the formatted string to the file
        f.write(formatted_line)

print(f"Successfully created {filename}")

Successfully created [04-09-26].txt
